# MTG Metagame GNN - Training Notebook

Train the Heterogeneous Graph Transformer (HGT) for MTG Standard metagame prediction on Google Colab with GPU acceleration.

**Runtime**: Select `Runtime > Change runtime type > T4 GPU` before running.

## 1. Setup

In [ ]:
# Mount Google Drive (project lives here)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import sys

# Set working directory to the project root in Drive
PROJECT_DIR = '/content/drive/MyDrive/mtg-graph'
os.chdir(PROJECT_DIR)

# Add project root to Python path so `from src.xxx` imports
# read directly from Drive (no pip install needed)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install third-party dependencies only (don't install our package —
# sys.path already points to the live Drive files)
!pip install -q torch-geometric sentence-transformers python-dotenv \
    beautifulsoup4 lxml spacy tqdm pyarrow

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

<![CDATA[## 2. Data Ingestion (Run Separately)

Data ingestion should be run **before** this notebook using the CLI utility:

```bash
python -m src.ingest_all                # Full pipeline (Phases 1-3)
python -m src.ingest_all --phase 1      # Phase 1 only (scrape)
python -m src.ingest_all --skip-scrape  # Phases 2-3 only (reuse data)
python -m src.ingest_all --validate-only # Just check existing files
```

The cells below assume `data/processed/graph.pt` already exists.]]>

In [ ]:
# Quick validation that data exists and graph is ready
from src.config import GRAPH_PATH, TOURNAMENTS_PARQUET, METAGAME_PARQUET, DECKLISTS_PARQUET
import pandas as pd

for name, path in [("Tournaments", TOURNAMENTS_PARQUET), ("Metagame", METAGAME_PARQUET),
                    ("Decklists", DECKLISTS_PARQUET), ("Graph", GRAPH_PATH)]:
    if path.exists():
        if path.suffix == ".parquet":
            rows = len(pd.read_parquet(path))
            print(f"  OK  {name}: {rows:,} rows")
        else:
            size_mb = path.stat().st_size / (1024 * 1024)
            print(f"  OK  {name}: {size_mb:.1f} MB")
    else:
        print(f"  MISSING  {name} — run: python -m src.ingest_all")

assert GRAPH_PATH.exists(), "Graph not found! Run data ingestion first."
print("\nData ready for training.")

## 3. Train Model (GPU-Accelerated)

In [ ]:
import os, time
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
from src.config import GRAPH_PATH, NUM_EPOCHS

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Quick graph validation before training
data = torch.load(GRAPH_PATH, weights_only=False)
print(f'\nNodes: {", ".join(f"{nt}={data[nt].x.shape[0]}" for nt in data.node_types)}')
print(f'Edge types: {len(data.edge_types)}')

problems = []
for et in data.edge_types:
    ei = data[et].edge_index
    src_type, rel, dst_type = et
    n_src, n_dst = data[src_type].x.shape[0], data[dst_type].x.shape[0]
    if ei.shape[1] == 0:
        problems.append(f'{et} has 0 edges')
        continue
    if ei[0].max().item() >= n_src:
        problems.append(f'{et} src index {ei[0].max().item()} >= {n_src}')
    if ei[1].max().item() >= n_dst:
        problems.append(f'{et} dst index {ei[1].max().item()} >= {n_dst}')

if problems:
    for p in problems:
        print(f'  PROBLEM: {p}')
    raise ValueError('Graph validation failed — fix before training')
else:
    print('All edge indices valid')
del data  # train() will reload it

from src.train import train

start_time = time.time()
result = train(device=device)
elapsed = time.time() - start_time
print(f'\nTraining complete in {elapsed:.1f}s ({elapsed/NUM_EPOCHS:.2f}s/epoch)')
print(f'Results directory: {result["run_dir"]}')

## 4. Training Curves

In [ ]:
from src.train import train

start_time = time.time()
result = train(device=device)
elapsed = time.time() - start_time
print(f'\nTraining complete in {elapsed:.1f}s ({elapsed/NUM_EPOCHS:.2f}s/epoch)')

## 5. Final Evaluation (Held-Out Test Set)

In [ ]:
import matplotlib.pyplot as plt

train_losses = result['train_losses']
val_losses = result['val_losses']
val_accs = result['val_accs']
best_epoch = result['best_epoch']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, len(train_losses) + 1), train_losses, alpha=0.4, label='Train (per epoch)')
val_epochs = [1] + list(range(5, NUM_EPOCHS + 1, 5))
ax1.plot(val_epochs[:len(val_losses)], val_losses, 'o-', label='Validation', markersize=3)
ax1.axvline(x=best_epoch, color='r', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training & Validation Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(val_epochs[:len(val_accs)], val_accs, 'o-', color='green', markersize=3)
ax2.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='Random baseline')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Validation Top-8 Accuracy')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Generate Dashboard

In [ ]:
from src.train import evaluate

eval_results = evaluate(
    model=result['model'],
    data=result['data'],
    arch_idx=result['arch_idx'],
    tourney_idx=result['tourney_idx'],
    top8_labels=result['top8_labels'],
    val_mask=result['val_mask'],
    test_mask=result['test_mask'],
    run_dir=result['run_dir'],
    model_path=result['model_path'],
    train_losses=result['train_losses'],
    val_losses=result['val_losses'],
    val_accs=result['val_accs'],
    best_epoch=result['best_epoch'],
)

print(f"\nResults saved to: {result['run_dir']}")

## 7. Generate Dashboard

In [ ]:
from src.visualize import main as generate_dashboard

generate_dashboard(
    run_dir=result['run_dir'],
    model_path=result['model_path'],
)

print(f"\nAll results in: {result['run_dir']}")
!ls -la "{result['run_dir']}"